# Exploration des sources TPCx-BB - Jour 3

UC-06 : chargement PySpark, schemas, volumes, valeurs manquantes et doublons des 4 sources retenues (ventes magasin, produits, magasins, temps).

In [1]:
import os, sys
os.environ["J3_RAW_MODE"] = "hdfs"          # lecture dans la zone Raw du data lake
os.environ["J3_DATA_DIR"] = "/home/amani/j3_data"
sys.path.insert(0, os.path.abspath("../scripts"))
from common import SOURCES, lire, spark_session

spark = spark_session("Jour3-exploration")
import pyspark, platform
print("Python :", platform.python_version())
print("PySpark :", pyspark.__version__)
print("Mode : local[2], VM Ubuntu 16.04")
for t, p in SOURCES.items():
    print(" ", t.ljust(12), p)

26/11/21 16:31:17 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/21 16:31:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/21 16:31:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Python : 3.10.18
PySpark : 3.5.5
Mode : local[2], VM Ubuntu 16.04
  store_sales  hdfs://localhost:9000/retail-lake/raw/tpcxbb/initial/store_sales.dat
  item         hdfs://localhost:9000/retail-lake/raw/tpcxbb/initial/item.dat
  date_dim     hdfs://localhost:9000/retail-lake/raw/tpcxbb/initial/date_dim.dat
  store        hdfs://localhost:9000/retail-lake/raw/nifi/store.dat


## Chargement des 4 sources
Fichiers `.dat` separes par `|`, sans en-tete : le schema est fourni explicitement (types deduits au Jour 1).

In [2]:
tables = ["store_sales", "item", "store", "date_dim"]
dfs = {t: lire(spark, t) for t in tables}
for t, d in dfs.items():
    print("###", t)
    d.printSchema()

### store_sales
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: double (nullable = true)
 |-- ss_list_price: double (nullable = true)
 |-- ss_sales_price: double (nullable = true)
 |-- ss_ext_discount_amt: double (nullable = true)
 |-- ss_ext_sales_price: double (nullable = true)
 |-- ss_ext_wholesale_cost: double (nullable = true)
 |-- ss_ext_list_price: double (nullable = true)
 |-- ss_ext_tax: double (nullable = true)
 |-- ss_coupon_amt: double (nullable = true)
 |-- ss_net_paid: double (nullable = true)
 |-- ss_net_paid_inc_tax: double (null

In [3]:
for t, d in dfs.items():
    print("###", t, "- echantillon")
    d.show(3, truncate=30)

### store_sales - echantillon


+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+

+---------+----------------+----------------+--------------+------------------------------+---------------+----------------+----------+------------------------------+----------+---------------------+-------------+-----------------+-------------+------------------------------+------+--------------------+-------+-------+-----------+------------+------------------------------+
|i_item_sk|       i_item_id|i_rec_start_date|i_rec_end_date|                   i_item_desc|i_current_price|i_wholesale_cost|i_brand_id|                       i_brand|i_class_id|              i_class|i_category_id|       i_category|i_manufact_id|                    i_manufact|i_size|       i_formulation|i_color|i_units|i_container|i_manager_id|                i_product_name|
+---------+----------------+----------------+--------------+------------------------------+---------------+----------------+----------+------------------------------+----------+---------------------+-------------+-----------------+-------------+-

26/11/21 16:31:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+----------------+----------------+--------------+----------------+------------+------------------+-------------+--------+--------------+-----------+-----------------+------------------------------+----------------+-------------+---------------+------------+--------------+---------------+---------------+-------------+--------------+------------+---------------+-------+-----+-------------+------------+----------------+
|s_store_sk|      s_store_id|s_rec_start_date|s_rec_end_date|s_closed_date_sk|s_store_name|s_number_employees|s_floor_space| s_hours|     s_manager|s_market_id|s_geography_class|                 s_market_desc|s_market_manager|s_division_id|s_division_name|s_company_id|s_company_name|s_street_number|  s_street_name|s_street_type|s_suite_number|      s_city|       s_county|s_state|s_zip|    s_country|s_gmt_offset|s_tax_precentage|
+----------+----------------+----------------+--------------+----------------+------------+------------------+-------------+--------+-

+---------+----------------+----------+-----------+----------+-------------+------+-----+-----+-----+-----+---------+----------------+-------------+----------+--------------+---------+---------+-------------------+-----------+----------+-------------+-------------+-------------+--------------+---------------+-----------------+--------------+
|d_date_sk|       d_date_id|    d_date|d_month_seq|d_week_seq|d_quarter_seq|d_year|d_dow|d_moy|d_dom|d_qoy|d_fy_year|d_fy_quarter_seq|d_fy_week_seq|d_day_name|d_quarter_name|d_holiday|d_weekend|d_following_holiday|d_first_dom|d_last_dom|d_same_day_ly|d_same_day_lq|d_current_day|d_current_week|d_current_month|d_current_quarter|d_current_year|
+---------+----------------+----------+-----------+----------+-------------+------+-----+-----+-----+-----+---------+----------------+-------------+----------+--------------+---------+---------+-------------------+-----------+----------+-------------+-------------+-------------+--------------+---------------+--

## Volumes et valeurs manquantes

In [4]:
from pyspark.sql import functions as F
for t, d in dfs.items():
    print("###", t, ":", d.count(), "lignes,", len(d.columns), "colonnes")
    nuls = [(c, d.filter(F.col(c).isNull()).count()) for c in d.columns]
    nuls = [(c, n) for c, n in nuls if n]
    print("   colonnes avec des nulls :", nuls if nuls else "aucune")

### store_sales : 667579 lignes, 23 colonnes


   colonnes avec des nulls : [('ss_cdemo_sk', 238), ('ss_hdemo_sk', 295), ('ss_addr_sk', 448), ('ss_promo_sk', 158)]


### item : 17820 lignes, 22 colonnes


   colonnes avec des nulls : [('i_rec_end_date', 17820), ('i_item_desc', 4), ('i_brand_id', 8), ('i_brand', 6)]
### store : 12 lignes, 29 colonnes


   colonnes avec des nulls : [('s_rec_end_date', 12)]


### date_dim : 109573 lignes, 28 colonnes


   colonnes avec des nulls : aucune


## Doublons sur les cles candidates

In [5]:
cles = {"store_sales": ["ss_item_sk", "ss_ticket_number"],
        "item": ["i_item_sk"], "store": ["s_store_sk"], "date_dim": ["d_date_sk"]}
for t, cols in cles.items():
    d = dfs[t]
    dup = d.count() - d.select(*cols).distinct().count()
    print(t, "| cle", cols, "| doublons :", dup)

store_sales | cle ['ss_item_sk', 'ss_ticket_number'] | doublons : 4136


item | cle ['i_item_sk'] | doublons : 0


store | cle ['s_store_sk'] | doublons : 0


date_dim | cle ['d_date_sk'] | doublons : 0


# Constats

- `store_sales` (667 579 lignes) : la cle couple (article, ticket) presente 4 136 doublons -
  un meme article peut figurer plusieurs fois sur un ticket, ce n'est pas une erreur (constat Jour 1).
- Valeurs manquantes : quelques centaines de nulls sur `ss_cdemo_sk`, `ss_hdemo_sk`,
  `ss_addr_sk`, `ss_promo_sk` (~0,1 % : informations demographiques/adresse/promo absentes).
  `ss_sold_date_sk` est toujours renseigne dans ce lot.
- `i_rec_end_date` (item) et `s_rec_end_date` (store) sont nulles pour 100 % des lignes :
  les fiches actives n'ont pas de date de fin, c'est normal.
- Les dimensions `item`, `store`, `date_dim` ont des cles candidates uniques (0 doublon).
- Les montants `ss_*` sont numeriques, les `*_sk` sont des identifiants entiers.

In [6]:
spark.stop()